In [0]:
from pyspark.sql.functions import col, count, month, year, to_date, concat, lit, when, last_day, lpad
from pyspark.sql import Window
import json
import os

# Paths (Updated to avoid DBFS issues)
output_path = "/Volumes/prd_datascience_onlinejobpostings/volumes/onlinejobpostings/joep_keuzenkamp/wind_vacancies_data_2017_18"
checkpoint_path = "/Volumes/prd_datascience_onlinejobpostings/volumes/onlinejobpostings/joep_keuzenkamp/checkpoints.json"

# Convert DBFS path to local path for JSON handling
local_checkpoint_path = "/dbfs" + checkpoint_path.lstrip("dbfs:")

# Load all files at once (parallel processing)
df = spark.read.parquet("dbfs:/mnt/light/data/world_bank_global_postings/version=3/2025/02/26/00_26_56/*.parquet")

# Filter for US & jobs posted after 2017-01-01
df_us = df.filter((col('nation_name') == "United States") & (col('posted') >= "2017-01-01") & (col('posted') < "2019-01-01"))

# Define keywords for solar-related jobs
keywords = [
    "wind energy", "wind farm", "wind generation", "wind park", "wind power", "wind turbine", 
    "wind energies", "wind farms", "wind generations", "wind parks", "wind powers", "wind turbines"
]
regex_values = "|".join(keywords)

# Filter for solar-related jobs
wind_df = df_us.filter(col("body").rlike(regex_values))

new_wind_df = (
    wind_df.select("posted", "employment_type", "laa_admin_area_2", "min_years_experience", "isco_08_1")
    .withColumn("month", lpad(month(col("posted")).cast("string"), 2, "0"))
    .withColumn("year", year(col("posted")))
    .groupBy("year", "month", "laa_admin_area_2", "min_years_experience", "isco_08_1")
    .agg(
        count(when(col("employment_type") == 1, True)).alias("full_time_count"),
        count(when(col("employment_type") == 2, True)).alias("part_time_count"),
        count("*").alias("total_count")
    )
    .withColumnRenamed("laa_admin_area_2", "county_id")
    .withColumnRenamed("isco_08_1", "career_type")
    .withColumn("start_date", to_date(concat(col("year"), lit("-"), col("month"), lit("-01")), "yyyy-MM-dd"))
    .withColumn("end_date", last_day(col("start_date")))
)

# Append data as CSV (instead of Parquet)
new_wind_df.write.format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("sep", ",") \
    .save(output_path)

In [0]:
# Load processed data
output_path = "/Volumes/prd_datascience_onlinejobpostings/volumes/onlinejobpostings/joep_keuzenkamp/wind_vacancies_data_2017_18"

df = spark.read.csv(output_path)

display(df)

_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9
year,month,county_id,min_years_experience,career_type,full_time_count,part_time_count,total_count,start_date,end_date
2017,09,US13121,6,3,2,0,2,2017-09-01,2017-09-30
2018,01,US48439,5,1,3,0,3,2018-01-01,2018-01-31
2017,09,US5119,4,2,2,0,2,2017-09-01,2017-09-30
2017,09,US48245,4,2,2,0,2,2017-09-01,2017-09-30
2018,01,US27047,3,7,1,0,1,2018-01-01,2018-01-31
2018,01,US39151,5,2,1,0,1,2018-01-01,2018-01-31
2017,01,US55025,null,5,1,3,4,2017-01-01,2017-01-31
2018,10,US56999,5,2,1,0,1,2018-10-01,2018-10-31
2018,10,US51117,2,7,1,0,1,2018-10-01,2018-10-31
